# Fase 4 · Prueba de concepto de Learning Analytics

**Herramienta:** Sistema de Analítica para la Planificación de Refuerzos Académicos.  
**Actor y decisión:** el ayudante de cátedra elige los temas y ejercicios de la próxima sesión semanal de refuerzo.  
**Pregunta descriptiva:** ¿qué contenidos presentan mayores dificultades según los ejercicios y las necesidades expresadas por los estudiantes?

Este notebook implementa el indicador central definido en *Fase 1 y Fase 2.pdf*. Usa **datos enteramente sintéticos**; ningún registro representa a estudiantes o cursos reales de la Universidad de Cuenca. Se ejecuta de arriba abajo en Google Colab con «Ejecutar todo», sin archivos externos. El código de generación también se entrega como `generar_datos_sinteticos.py`.

La visualización Plotly reproduce los elementos principales del [wireframe de priorización](../docs/03_diseno_herramienta.pdf): ranking por tema, Dₒ, Dₚ, Pᵣ y cobertura. La variante de Fase 4 usa las cifras calculadas aquí. Los planes y el seguimiento interactivos se demuestran aparte en `app.py`; este notebook verifica el indicador central.

## 1. Entorno y datos

La ventana de análisis es **del 9 al 15 de octubre de 2026**, antes de la sesión del 16 de octubre. Se simulan un curso, cinco temas y 64 estudiantes con códigos seudónimos. Las tablas tienen esta granularidad:

| Tabla | Unidad de cada fila | Uso |
|---|---|---|
| `temas` | tema de un curso | nombres para mostrar |
| `resultados` | estudiante–actividad–pregunta–intento | dificultad observada |
| `encuestas` | estudiante–tema–respuesta | dificultad percibida |

El generador usa semilla fija. Las probabilidades de acierto y de respuesta son parámetros de simulación, no estimaciones de la UCuenca. Se añade a propósito un tema sin encuesta, un segundo intento, duplicados, datos fuera de período y valores inválidos para verificar las reglas del diccionario.

In [1]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("plotly") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "plotly>=5.18,<7"])

from IPython.display import display
import plotly.graph_objects as go
from plotly.subplots import make_subplots

### Script documentado de generación sintética

La siguiente celda contiene íntegro el generador necesario para ejecutar el notebook en Colab sin subir archivos adicionales. La versión `.py` incluida en el entregable permite regenerar las tres tablas CSV por separado.

In [2]:
"""Datos ficticios y reproducibles para la prueba de concepto de refuerzos.

No contiene información de estudiantes ni cursos reales de la UCuenca.
La semilla fija permite repetir exactamente las cifras del notebook.
"""

from pathlib import Path

import numpy as np
import pandas as pd


SEMILLA = 20261016
CURSO_ID = "CURSO_DEMO_01"
ASIGNATURA_ID = "ASIGNATURA_DEMO"
INICIO = "2026-10-09"
FIN = "2026-10-15"


def generar_datos(semilla=SEMILLA):
    """Devuelve catálogo de temas, resultados por intento y encuestas por respuesta.

    Simula 64 estudiantes seudónimos, un curso y una ventana de siete días.
    Los puntajes son aciertos de 10; la probabilidad de acierto varía por tema.
    Se agregan duplicados, intentos posteriores y algunos errores para probar
    las reglas de limpieza. T05 no tiene encuesta: sirve para comprobar que la
    prioridad queda ausente cuando falta uno de sus dos componentes.
    """
    rng = np.random.default_rng(semilla)
    temas = pd.DataFrame(
        [
            ("T01", "Probabilidad condicional", 0.46, [0.03, 0.07, 0.18, 0.34, 0.38]),
            ("T02", "Distribuciones", 0.67, [0.12, 0.19, 0.28, 0.26, 0.15]),
            ("T03", "Intervalos de confianza", 0.75, [0.18, 0.27, 0.31, 0.16, 0.08]),
            ("T04", "Pruebas de hipótesis", 0.53, [0.05, 0.10, 0.20, 0.35, 0.30]),
            ("T05", "Regresión lineal", 0.63, None),
        ],
        columns=["tema_id", "tema", "prob_acierto_simulada", "prob_encuesta_simulada"],
    )
    estudiantes = [f"E{i:03d}" for i in range(1, 65)]
    dias = pd.date_range(INICIO, FIN, freq="D")
    resultados = []
    encuestas = []

    for tema in temas.itertuples(index=False):
        for estudiante in estudiantes:
            if rng.random() < 0.84:
                actividad = f"ACT_{tema.tema_id}"
                for numero_pregunta in range(1, 4):
                    fecha = rng.choice(dias)
                    base = dict(
                        asignatura_id=ASIGNATURA_ID,
                        curso_id=CURSO_ID,
                        estudiante_id=estudiante,
                        actividad_id=actividad,
                        tipo_actividad="ejercicio",
                        pregunta_id=f"{tema.tema_id}_P{numero_pregunta}",
                        tema_id=tema.tema_id,
                        puntaje_maximo=10,
                    )
                    resultados.append(
                        {**base, "numero_intento": 1, "fecha_intento": str(pd.Timestamp(fecha).date()),
                         "puntaje_obtenido": int(rng.binomial(10, tema.prob_acierto_simulada))}
                    )
                    if rng.random() < 0.12:
                        resultados.append(
                            {**base, "numero_intento": 2, "fecha_intento": str(dias[-1].date()),
                             "puntaje_obtenido": int(rng.binomial(10, min(0.95, tema.prob_acierto_simulada + 0.15)))}
                        )
            if tema.prob_encuesta_simulada is not None and rng.random() < 0.72:
                valor = int(rng.choice(np.arange(1, 6), p=tema.prob_encuesta_simulada))
                encuestas.append(
                    {"curso_id": CURSO_ID, "estudiante_id": estudiante, "tema_id": tema.tema_id,
                     "fecha_encuesta": str(pd.Timestamp(rng.choice(dias)).date()), "dificultad_percibida": str(valor)}
                )

    # Casos controlados para verificar la limpieza, sin alterar el diseño del indicador.
    resultados.extend([resultados[0].copy(), resultados[1].copy()])
    resultados.append({**resultados[2], "puntaje_obtenido": 12})
    resultados.append({**resultados[3], "estudiante_id": None})
    resultados.append({**resultados[4], "fecha_intento": "2026-10-08"})
    encuestas.append(encuestas[0].copy())
    encuestas.append({**encuestas[1], "dificultad_percibida": "6"})
    encuestas.append({**encuestas[2], "fecha_encuesta": "2026-10-08"})
    encuestas.append({**encuestas[3], "dificultad_percibida": "No he trabajado este tema"})
    encuestas[4]["fecha_encuesta"] = "2026-10-12"
    encuestas.append({**encuestas[4], "fecha_encuesta": FIN, "dificultad_percibida": "5"})

    return temas.drop(columns=["prob_acierto_simulada", "prob_encuesta_simulada"]), pd.DataFrame(resultados), pd.DataFrame(encuestas)

In [3]:
datos_dir = Path("datos")
datos_dir.mkdir(exist_ok=True)
tablas_generadas = dict(zip(("temas", "resultados", "encuestas"), generar_datos()))
for nombre, tabla in tablas_generadas.items():
    tabla.to_csv(datos_dir / f"{nombre}.csv", index=False)

temas = pd.read_csv(datos_dir / "temas.csv")
resultados_brutos = pd.read_csv(datos_dir / "resultados.csv")
encuestas_brutas = pd.read_csv(datos_dir / "encuestas.csv")

print(f"Semilla: {SEMILLA} | Ventana: {INICIO} a {FIN}")
print(f"Cursos: {resultados_brutos['curso_id'].nunique()} | Estudiantes simulados: 64 | Temas: {len(temas)}")
print(f"Resultados por intento: {len(resultados_brutos)} | Respuestas de encuesta: {len(encuestas_brutas)}")
print("Tablas CSV generadas y cargadas desde:", datos_dir)
print("Los identificadores de curso y asignatura son ficticios.")
display(temas)
display(resultados_brutos.head(3))
display(encuestas_brutas.head(3))

Semilla: 20261016 | Ventana: 2026-10-09 a 2026-10-15
Cursos: 1 | Estudiantes simulados: 64 | Temas: 5
Resultados por intento: 911 | Respuestas de encuesta: 185
Tablas CSV generadas y cargadas desde: datos
Los identificadores de curso y asignatura son ficticios.


,tema_id,tema
0,T01,Probabilidad condicional
1,T02,Distribuciones
2,T03,Intervalos de confianza
3,T04,Pruebas de hipótesis
4,T05,Regresión lineal


,asignatura_id,curso_id,estudiante_id,actividad_id,tipo_actividad,pregunta_id,tema_id,puntaje_maximo,numero_intento,fecha_intento,puntaje_obtenido
0,ASIGNATURA_DEMO,CURSO_DEMO_01,E001,ACT_T01,ejercicio,T01_P1,T01,10,1,2026-10-11,5
1,ASIGNATURA_DEMO,CURSO_DEMO_01,E001,ACT_T01,ejercicio,T01_P2,T01,10,1,2026-10-12,6
2,ASIGNATURA_DEMO,CURSO_DEMO_01,E001,ACT_T01,ejercicio,T01_P3,T01,10,1,2026-10-12,5


,curso_id,estudiante_id,tema_id,fecha_encuesta,dificultad_percibida
0,CURSO_DEMO_01,E002,T01,2026-10-10,3
1,CURSO_DEMO_01,E004,T01,2026-10-11,2
2,CURSO_DEMO_01,E006,T01,2026-10-09,4


## 2. Reglas de limpieza

Se usan solo registros del mismo curso y de la ventana de siete días. Para resultados, se exige estudiante, curso, actividad, pregunta, tema, fecha y puntajes válidos. Los duplicados exactos se eliminan; si un mismo intento tiene valores contradictorios, se excluye ese intento. Después se toma el **primer intento válido** de cada estudiante por actividad y pregunta. Para encuestas se conserva la **última respuesta válida** de cada estudiante por tema; «No he trabajado este tema» y respuestas fuera de 1 a 5 se excluyen. La ausencia de datos nunca se transforma en cero.

El orden de filtros y el número de filas retiradas se muestran para que el tratamiento sea auditable.

In [4]:
inicio = pd.Timestamp(INICIO)
fin = pd.Timestamp(FIN)
calidad = []

def filtrar(tabla, nombre, motivo, mascara_valida):
    retiradas = int((~mascara_valida).sum())
    calidad.append({"tabla": nombre, "regla": motivo, "filas_retiradas": retiradas})
    return tabla.loc[mascara_valida].copy()

r = resultados_brutos.copy()
r["fecha_intento"] = pd.to_datetime(r["fecha_intento"], errors="coerce")
for col in ["puntaje_obtenido", "puntaje_maximo", "numero_intento"]:
    r[col] = pd.to_numeric(r[col], errors="coerce")

r = filtrar(r, "resultados", "fecha dentro del período", r["fecha_intento"].between(inicio, fin))
ids_r = ["asignatura_id", "curso_id", "estudiante_id", "actividad_id", "pregunta_id", "tema_id"]
r = filtrar(r, "resultados", "identificadores presentes", r[ids_r].notna().all(axis=1))
r = filtrar(r, "resultados", "puntajes entre 0 y máximo positivo",
            r["puntaje_maximo"].gt(0) & r["puntaje_obtenido"].ge(0)
            & r["puntaje_obtenido"].le(r["puntaje_maximo"]))
r = filtrar(r, "resultados", "número de intento entero positivo",
            r["numero_intento"].ge(1) & r["numero_intento"].mod(1).eq(0))
r = filtrar(r, "resultados", "duplicados exactos", ~r.duplicated())

clave_intento = ["curso_id", "estudiante_id", "actividad_id", "pregunta_id", "numero_intento"]
r = filtrar(r, "resultados", "intentos contradictorios", ~r.duplicated(clave_intento, keep=False))
assert r.groupby(["curso_id", "actividad_id", "pregunta_id"])["tema_id"].nunique().max() == 1, "Una pregunta aparece en varios temas"

clave_pregunta = ["curso_id", "estudiante_id", "actividad_id", "pregunta_id"]
r = r.sort_values(["numero_intento", "fecha_intento"], kind="stable")
r = filtrar(r, "resultados", "intentos posteriores al primero válido", ~r.duplicated(clave_pregunta))
r["dificultad_respuesta"] = (1 - r["puntaje_obtenido"] / r["puntaje_maximo"]) * 100

e = encuestas_brutas.copy()
e["fecha_encuesta"] = pd.to_datetime(e["fecha_encuesta"], errors="coerce")
e["dificultad_num"] = pd.to_numeric(e["dificultad_percibida"], errors="coerce")
e = filtrar(e, "encuestas", "fecha dentro del período", e["fecha_encuesta"].between(inicio, fin))
e = filtrar(e, "encuestas", "identificadores presentes", e[["curso_id", "estudiante_id", "tema_id"]].notna().all(axis=1))
e = filtrar(e, "encuestas", "escala válida de 1 a 5 (incluye 'No he trabajado' como excluido)",
            e["dificultad_num"].between(1, 5) & e["dificultad_num"].mod(1).eq(0))
e = filtrar(e, "encuestas", "duplicados exactos", ~e.duplicated())
clave_encuesta = ["curso_id", "estudiante_id", "tema_id"]
e = e.sort_values("fecha_encuesta", kind="stable")
e = filtrar(e, "encuestas", "respuestas anteriores a la última válida", ~e.duplicated(clave_encuesta, keep="last"))

display(pd.DataFrame(calidad))
print(f"Resultados seleccionados: {len(r)} | Encuestas seleccionadas: {len(e)}")

,tabla,regla,filas_retiradas
0,resultados,fecha dentro del período,1
1,resultados,identificadores presentes,1
2,resultados,puntajes entre 0 y máximo positivo,1
3,resultados,número de intento entero positivo,0
4,resultados,duplicados exactos,2
5,resultados,intentos contradictorios,0
6,resultados,intentos posteriores al primero válido,96
7,encuestas,fecha dentro del período,1
8,encuestas,identificadores presentes,0
9,encuestas,escala válida de 1 a 5 (incluye 'No he trabaja...,2


Resultados seleccionados: 810 | Encuestas seleccionadas: 180


## 3. Indicadores por curso y tema

Se respetan las fórmulas acordadas en la Fase 2:

\(D_o = 100\,\operatorname{promedio}_i(1 - P_i/P_{max,i})\), dando **igual peso a cada respuesta válida**.  
\(D_p = 100\,\#(respuestas\ 4\ o\ 5)/\#(respuestas\ válidas)\), una respuesta por estudiante y tema.  
\(P_r = 0.5D_o + 0.5D_p\), solamente si ambos existen.

El índice ordena temas para apoyar la planificación. No representa probabilidad de fracaso, diagnóstico individual ni tiempo de clase recomendado. Junto a cada valor se muestran los tamaños de ambas fuentes, porque sus participantes pueden diferir.

In [5]:
observada = (r.groupby(["curso_id", "tema_id"], as_index=False)
    .agg(dificultad_observada=("dificultad_respuesta", "mean"),
         n_estudiantes_academicos=("estudiante_id", "nunique"),
         n_respuestas_evaluadas=("pregunta_id", "size"),
         n_preguntas_distintas=("pregunta_id", "nunique")))

e["dificultad_alta"] = e["dificultad_num"].isin([4, 5]).astype(int)
percibida = (e.groupby(["curso_id", "tema_id"], as_index=False)
    .agg(dificultad_percibida_pct=("dificultad_alta", lambda x: x.mean() * 100),
         n_estudiantes_encuesta=("estudiante_id", "nunique")))

catalogo = temas.assign(curso_id=CURSO_ID)
resumen = (catalogo.merge(observada, on=["curso_id", "tema_id"], how="left")
    .merge(percibida, on=["curso_id", "tema_id"], how="left"))
ambos = resumen[["dificultad_observada", "dificultad_percibida_pct"]].notna().all(axis=1)
resumen["prioridad_refuerzo"] = (
    0.5 * resumen["dificultad_observada"]
    + 0.5 * resumen["dificultad_percibida_pct"]
)
resumen["estado"] = np.where(ambos, "Calculable", "Información incompleta")
resumen = resumen.sort_values("prioridad_refuerzo", ascending=False, na_position="last").reset_index(drop=True)

assert resumen["prioridad_refuerzo"].dropna().between(0, 100).all()
assert resumen.loc[resumen["estado"].eq("Información incompleta"), "prioridad_refuerzo"].isna().all()
assert resumen.loc[resumen["tema_id"].eq("T05"), "prioridad_refuerzo"].isna().all()
assert r.duplicated(clave_pregunta).sum() == 0
assert e.duplicated(clave_encuesta).sum() == 0

display(resumen.round(2))
print("Tema con mayor prioridad calculable:", resumen.loc[resumen["prioridad_refuerzo"].first_valid_index(), "tema"])
print("Prioridades incompletas:", int(resumen["prioridad_refuerzo"].isna().sum()))

,tema_id,tema,curso_id,dificultad_observada,n_estudiantes_academicos,n_respuestas_evaluadas,n_preguntas_distintas,dificultad_percibida_pct,n_estudiantes_encuesta,prioridad_refuerzo,estado
0,T01,Probabilidad condicional,CURSO_DEMO_01,52.55,55,165,3,72.73,44.0,62.64,Calculable
1,T04,Pruebas de hipótesis,CURSO_DEMO_01,45.39,55,165,3,68.09,47.0,56.74,Calculable
2,T02,Distribuciones,CURSO_DEMO_01,34.23,52,156,3,44.90,49.0,39.56,Calculable
3,T03,Intervalos de confianza,CURSO_DEMO_01,24.13,50,150,3,20.00,40.0,22.07,Calculable
4,T05,Regresión lineal,CURSO_DEMO_01,37.30,58,174,3,NaN,NaN,NaN,Información incompleta


Tema con mayor prioridad calculable: Probabilidad condicional
Prioridades incompletas: 1


## 4. Vista tipo tablero para el ayudante

Propuesta funcional para una pantalla de planificación: **ranking de prioridad**, comparación de sus dos componentes y **tabla con cobertura**. La fila «Información incompleta» permanece visible y no recibe un valor inventado. El ayudante revisaría los primeros temas junto con los tamaños de muestra y elegiría contenidos y ejercicios para la próxima sesión. El tablero no impone un umbral ni toma la decisión por él.

Esta composición corresponde a la primera pantalla del wireframe de Fase 3. El HTML generado permite abrir el tablero sin ejecutar Python. La aplicación Streamlit del repositorio demuestra la acción de planificar y registrar seguimiento.

In [6]:
def formato_pct(valor):
    return "Sin datos" if pd.isna(valor) else f"{valor:.1f}%"

orden = resumen["tema"].tolist()
prioridad = resumen.dropna(subset=["prioridad_refuerzo"])
fig = make_subplots(
    rows=2, cols=2,
    specs=[[{"type": "xy"}, {"type": "xy"}], [{"type": "table", "colspan": 2}, None]],
    subplot_titles=("Prioridad de refuerzo por tema", "Componentes del indicador", "Cobertura y resultados"),
    row_heights=[0.56, 0.44], vertical_spacing=0.14,
)
fig.add_trace(go.Bar(
    x=prioridad["prioridad_refuerzo"], y=prioridad["tema"], orientation="h",
    name="Prioridad", marker_color="#315caa",
    text=prioridad["prioridad_refuerzo"].map(lambda x: f"{x:.1f}"), textposition="outside",
    hovertemplate="%{y}<br>Prioridad: %{x:.1f}/100<extra></extra>",
), row=1, col=1)
for columna, etiqueta, color in [
    ("dificultad_observada", "Observada", "#d97732"),
    ("dificultad_percibida_pct", "Percibida", "#49998b"),
]:
    fig.add_trace(go.Bar(
        x=resumen[columna], y=resumen["tema"], orientation="h",
        name=etiqueta, marker_color=color,
        hovertemplate=f"%{{y}}<br>{etiqueta}: %{{x:.1f}}%<extra></extra>",
    ), row=1, col=2)

columnas_tabla = [
    ("Tema", resumen["tema"].tolist()),
    ("Do", [formato_pct(x) for x in resumen["dificultad_observada"]]),
    ("Dp", [formato_pct(x) for x in resumen["dificultad_percibida_pct"]]),
    ("Pr", [formato_pct(x) for x in resumen["prioridad_refuerzo"]]),
    ("Est. acad.", ["—" if pd.isna(x) else str(int(x)) for x in resumen["n_estudiantes_academicos"]]),
    ("Respuestas", ["—" if pd.isna(x) else str(int(x)) for x in resumen["n_respuestas_evaluadas"]]),
    ("Preguntas", ["—" if pd.isna(x) else str(int(x)) for x in resumen["n_preguntas_distintas"]]),
    ("Encuestas", ["—" if pd.isna(x) else str(int(x)) for x in resumen["n_estudiantes_encuesta"]]),
    ("Estado", resumen["estado"].tolist()),
]
fig.add_trace(go.Table(
    columnwidth=[210, 65, 65, 65, 75, 90, 80, 80, 145],
    header=dict(values=[c[0] for c in columnas_tabla], fill_color="#e8eef8", align="left", font=dict(size=12)),
    cells=dict(values=[c[1] for c in columnas_tabla], fill_color="white", align="left", height=30, font=dict(size=11)),
), row=2, col=1)
fig.update_layout(
    title=f"Planificación del refuerzo · {CURSO_ID} · {INICIO} a {FIN}",
    width=1450, height=850, barmode="group", template="plotly_white",
    legend=dict(orientation="h", y=1.05, x=0.63), margin=dict(l=90, r=45, t=105, b=45),
)
fig.update_xaxes(range=[0, 108], title_text="Puntos / 100", row=1, col=1)
fig.update_xaxes(range=[0, 100], title_text="Porcentaje", row=1, col=2)
fig.update_yaxes(autorange="reversed", row=1, col=1)
fig.update_yaxes(autorange="reversed", row=1, col=2)
fig.show()
fig.write_html("tablero_prueba_concepto.html", include_plotlyjs=True, full_html=True)
print("Tablero guardado en tablero_prueba_concepto.html")

Tablero guardado en tablero_prueba_concepto.html


## 5. Lo que este prototipo no puede decir todavía

1. **Validez técnica.** Que el índice sea calculable no demuestra que mida necesidades reales de refuerzo. Hay que contrastarlo con resultados posteriores o juicio docente, evaluar la ponderación 50/50 y revisar si las preguntas de los temas son comparables.
2. **Impacto pedagógico.** Los datos sintéticos no demuestran que planificar con este tablero mejore comprensión, asistencia o notas. El plan de validación del grupo propone una comparación con grupos y períodos adecuados; no se ha ejecutado y se conserva fuera de este repositorio.
3. **Usabilidad y adopción.** No se ha probado si ayudantes y docentes entienden el índice, las coberturas ni la etiqueta «Información incompleta». Hacen falta pruebas con usuarios.
4. **Representatividad y sesgos.** Una encuesta voluntaria puede atraer más a quienes tienen dificultades; quienes faltan en resultados o encuestas pueden diferir. Las dos fuentes no contienen necesariamente a los mismos estudiantes.
5. **Privacidad y operación.** Con datos reales habría que autorizar acceso, seudonimizar, limitar las vistas por rol y evitar desgloses de grupos pequeños. No se ha implementado la conexión con eVirtual. La aplicación de demostración registra acciones solo en la sesión del navegador.

**Modelo predictivo:** no aplica en esta fase porque la pregunta y el indicador de las Fases 1 y 2 son descriptivos. No se generan predicciones individuales.

**Uso de IA generativa:** se utilizó OpenAI Codex para redactar y programar este prototipo. La ejecución reproducible y las comprobaciones de rangos, duplicados y datos faltantes se verifican con las celdas anteriores. La [declaración completa](../docs/07_uso_ia.pdf) requiere revisión final del grupo.